<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 8B: *Fire Ignition Feature Ablation*

## Metadata

---
### Contents  
> 1. **
> 2. **
> 3. **
---
### Notes
---
### Inputs
---
### Outputs 
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core libraries
import pandas as pd
import numpy as np
import json

# Modeling libraries
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import train_test_split

---

##  Load Files

In [3]:
X_ignition = pd.read_csv('../data/processed/X_ignition.csv')
y_ignition = pd.read_csv('../data/processed/y_ignition.csv').squeeze()  # Load as Series      
details_ignition = pd.read_csv('../data/processed/details_ignition.csv')

pal_details = pd.read_csv('../data/processed/pal_details.csv')
pal_X = pd.read_csv('../data/processed/pal_X.csv')
pal_y = pd.read_csv('../data/processed/pal_y.csv')

best_strategy = pd.read_csv('../data/processed/ignition_best_strategy.csv')

with open('model_parameters_ignition.json', 'r') as f:
    model_parameters = json.load(f)

with open('feature_sets.json', 'r') as f:
    feature_sets = json.load(f)

## Subset

In [4]:
reform = pd.concat([X_ignition,y_ignition], axis=1)
subset = subset_df(reform, 'Target_Ignition', 500)

y = subset['Target_Ignition']
X = subset.drop(columns='Target_Ignition')

In [5]:
X_rf, y_rf = apply_balancing('RF', best_strategy, X, y)
X_xgb, y_xgb = apply_balancing('XGB', best_strategy, X, y)

## Build Models

In [6]:
RF_parameters = model_parameters['Random Forest']
XGB_parameters = model_parameters['XGBoost']

# Build tuned models
ignition_xbg = xgb.XGBClassifier(**XGB_parameters)
ignition_rf = RandomForestClassifier(**RF_parameters)

display(RF_parameters)
display(XGB_parameters)

{'n_estimators': 150,
 'max_depth': 15,
 'min_samples_split': 2,
 'max_features': 'log2',
 'class_weight': 'balanced'}

{'objective': 'multi:softmax',
 'num_class': 3,
 'n_estimators': 100,
 'max_depth': 4,
 'learning_rate': 0.1,
 'verbosity': 0}

In [7]:
models = {
    "XGB": ignition_xbg, 
    "RF": ignition_rf
}

## SHAP

In [8]:
xgb_top_10 = get_shap(ignition_xbg, X_xgb, y_xgb)
xgb_top_10


,0,1
0,intermix_zone,0.290884
1,total_housing,0.254696
2,total_population,0.245733
3,road_length_meters,0.227775
4,1000-hour Dead Fuel Moisture,0.184763
5,power_line_density_x_total_housing,0.179165
6,Season_Summer,0.165355
7,Standardized Precipitation Evapotranspiration ...,0.127502
8,Wind Speed_x_slope_max,0.124560
9,influence_zone,0.118623


In [9]:
rf_top_10 = get_shap(ignition_rf, X_rf, y_rf)
rf_top_10 


 41%|========            | 373/900 [00:11<00:15]       


 46%|=========           | 414/900 [00:12<00:14]       


 51%|==========          | 455/900 [00:13<00:12]       


 55%|===========         | 496/900 [00:14<00:11]       


 60%|============        | 537/900 [00:15<00:10]       

...


 85%|=================   | 766/900 [00:21<00:03]       


 90%|==================  | 807/900 [00:22<00:02]       


 94%|=================== | 848/900 [00:23<00:01]       


 99%|===================| 890/900 [00:24<00:00]       

,0,1
0,power_line_density_x_total_housing,0.016990
1,total_population,0.016850
2,population_density,0.015548
3,total_housing,0.015207
4,intermix_zone,0.014872
5,road_length_meters,0.014691
6,housing_density,0.014580
7,influence_zone,0.013735
8,interface_zone,0.012225
9,road_density,0.011000


## Ablation

In [10]:
for set_name, columns in feature_sets.items():
    print(f"{set_name}: {columns}")
    print("\n")

Water Demand: ['Actual Evapotranspiration', 'Solar Radiation', 'Solar Radiation 7 Day Avg', 'Daily Minimum Air Temperature', 'Daily Maximum Air Temperature', 'Daily Maximum Air Temperature 7 Day Avg', 'Daily Minimum Air Temperature 7 Day Avg', 'Vapor Pressure Deficit', 'Vapor Pressure Deficit 7 Day Avg', 'Wind Speed', 'Wind Speed 7 Day Avg']


Water Supply: ['Precipitation', 'Precipitation 7 Day Avg', 'Maximum Relative Humidity', 'Minimum Relative Humidity', 'Maximum Relative Humidity 7 Day Avg', 'Minimum Relative Humidity 7 Day Avg', 'Specific Humidity', '100-hour Dead Fuel Moisture', '1000-hour Dead Fuel Moisture']


Water Supply Indexes: ['Standardized Precipitation Index 30-Day', 'Standardized Precipitation Index 180-Day', 'Standardized Precipitation Evapotranspiration Index 30-Day', 'Standardized Precipitation Evapotranspiration Index 90-Day', 'Standardized Precipitation Evapotranspiration Index 180-Day', 'Palmer Drought Severity Index']


Fire Danger: ['Burning Index', 'Energy Re

In [11]:
compare = []

for set_name, set_list in feature_sets.items():
    for model_name, model in models.items():
        print("Testing " + f"{model_name}: {set_name}")
        X_section = X.drop(columns = set_list)
        compare.append(compare_model(model, X_section, y, best_strategy,
                                     name = model_name, test_set = set_name))

Testing XGB: Water Demand


Testing RF: Water Demand


Testing XGB: Water Supply


Testing RF: Water Supply


Testing XGB: Water Supply Indexes


...

Testing RF: Interactions


Testing XGB: Wind Slope


Testing RF: Wind Slope


Testing XGB: Others


Testing RF: Others


In [12]:
comparisons = pd.DataFrame(compare)
comparisons

,Test Set,Model,Weighted F1,Macro F1,High Risk Recall
0,Water Demand,XGB,0.616657,0.625484,0.642857
1,Water Demand,RF,0.623270,0.629877,0.660714
2,Water Supply,XGB,0.607919,0.618183,0.616071
3,Water Supply,RF,0.628192,0.635072,0.616071
4,Water Supply Indexes,XGB,0.598293,0.609220,0.598214
5,Water Supply Indexes,RF,0.626088,0.633764,0.607143
6,Fire Danger,XGB,0.618323,0.628377,0.607143
7,Fire Danger,RF,0.624595,0.632631,0.616071
8,Social,XGB,0.630368,0.639953,0.642857
9,Social,RF,0.632421,0.638998,0.633929
